# BEE 4750 Homework 5: Mixed Integer and Stochastic Programming

**Name**: Evan Wu

**ID**: 5839048

> **Due Date**
>
> Thursday, 12/04/24, 9:00pm

## Overview

### Instructions

-   In Problem 1, you will use mixed integer programming to solve a
    waste load allocation problem.
-   In Problem 2, you will formulate a stochastic optimization problem.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [1]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `c:\Users\evant\OneDrive\Documents\GitHub\hw5-evatw`
┌ Warning: The active manifest file has dependencies that were resolved with a different julia version (1.11.5). Unexpected behavior may occur.
└ @ nothing C:\Users\evant\OneDrive\Documents\GitHub\hw5-evatw\Manifest.toml:0
┌ Warning: The active manifest file has dependencies that were resolved with a different julia version (1.11.5). Unexpected behavior may occur.
└ @ nothing C:\Users\evant\OneDrive\Documents\GitHub\hw5-evatw\Manifest.toml:0
┌ Warning: The project dependencies or compat requirements have changed since the manifest was last resolved.
│ It is recommended to `Pkg.resolve()` or consider `Pkg.update()` if necessary.
└ @ Pkg.API C:\Users\evant\.julia\juliaup\julia-1.12.1+0.x64.w64.mingw32\share\julia\stdlib\v1.12\Pkg\src\API.jl:1227
┌ Warning: The project dependencies or compat requirements have changed since the manifest was last resolved.
│ It is recommended to `Pkg.resolve()` or consider `Pkg.upd

In [2]:
using JuMP
using HiGHS
using DataFrames
using GraphRecipes
using Plots
using Measures
using MarkdownTables

## Problems (Total: 30 Points)

### Problem 1 (24 points)

Three cities are developing a coordinated municipal solid waste (MSW)
disposal plan. Three disposal alternatives are being considered: a
landfill (LF), a materials recycling facility (MRF), and a
waste-to-energy facility (WTE). The capacities of these facilities and
the fees for operation and disposal are provided below.

-   **LF**: Capacity 200 Mg, fixed cost \$2000/day, tipping cost
    \$50/Mg;
-   **MRF**: Capacity 350 Mg, fixed cost \$1500/day, tipping cost
    \$7/Mg, recycling cost \$40/Mg recycled;
-   **WTE**: Capacity 210 Mg, fixed cost \$2500/day, tipping cost
    \$60/Mg;

The MRF recycling rate is 40%, and the ash fraction of non-recycled
waste is 16% and of recycled waste is 14%. Transportation costs are
\$1.5/Mg-km, and the relative distances between the cities and
facilities are provided in the table below.

| **City/Facility** | **Landfill (km)** | **MRF (km)** | **WTE (km)** |
|:-----------------:|:-----------------:|:------------:|:------------:|
|         1         |         5         |      30      |      15      |
|         2         |        15         |      25      |      10      |
|         3         |        13         |      45      |      20      |
|        LF         |        \-         |      32      |      18      |
|        MRF        |        32         |      \-      |      15      |
|        WTE        |        18         |      15      |      \-      |

The fixed costs associated with the disposal options are incurred only
if the particular disposal option is implemented. The three cities
produce 100, 90, and 120 Mg/day of solid waste, respectively, with the
composition provided in the table below.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 |
| Paper & Cardboard | 40 | 7 | 55 |
| Plastics | 5 | 5 | 15 |
| Textiles | 3 | 10 | 10 |
| Rubber, Leather | 2 | 15 | 0 |
| Wood | 5 | 2 | 30 |
| Yard Wastes | 18 | 2 | 40 |
| Glass | 4 | 100 | 60 |
| Ferrous | 2 | 100 | 75 |
| Aluminum | 2 | 100 | 80 |
| Other Metal | 1 | 100 | 50 |
| Miscellaneous | 3 | 70 | 0 |

The information in the above table will help you determine the overall
recycling and ash fractions. Note that the recycling residuals, which
may be sent to either landfill or the WTE, have different ash content
than the ash content of the original MSW. You will need to determine
these fractions to construct your mass balance constraints.

**Reminder**: Use `round(x; digits=n)` to report values to the
appropriate precision!

#### Problem 1.1

Based on the information above, calculate the overall recycling and ash
fractions for the waste produced by each city.

Recycling fraction would be the percentage of total mass recycled with MRF, ash fraction would be the remaining ash from the non-recycled component w.r.t the total mass.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) | **Non-recycled amount** (%) | **Recycle fraction** (%) | **Ash fraction** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|:---------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 | 15 | 0 | 1.20
| Paper & Cardboard | 40 | 7 | 55 | 18 | 22 | 1.26
| Plastics | 5 | 5 | 15 | 4.25 | 0.75 | 0.21
| Textiles | 3 | 10 | 10 | 2.70 | 0.30 | 0.27
| Rubber, Leather | 2 | 15 | 0 | 2 | 0 | 0.3
| Wood | 5 | 2 | 30 | 3.5 | 1.5 | 0.07
| Yard Wastes | 18 | 2 | 40 | 10.8 | 7.2 | 0.216
| Glass | 4 | 100 | 60 | 1.6 | 2.4 | 1.6 
| Ferrous | 2 | 100 | 75 | 0.5 | 1.5 | 0.5 
| Aluminum | 2 | 100 | 80 | 0.4 | 1.6 | 0.4 
| Other Metal | 1 | 100 | 50 | 0.5 | 0.5 | 0.5
| Miscellaneous | 3 | 70 | 0 | 3 | 0 | 2.1

The total recycle fraction from each city would be 37.75 or rounded to 38 (summing together all the recycle fraction values), while the total ash fraction would be 8.62 or rounded to 9 (summing together all the ash fraction values).

#### Problem 1.2

What are the decision variables for your optimization problem? Provide
notation and variable meaning.

The decision variables for this optimization problem would be the mass of waste going into each facility f  from each city i in Mg. There is also the mass of recycle R from each city to be processed by the MRF and whether a binary decision variable to check if a facility is on. 

$$ x_{i,f}, R_i, y_i $$

#### Problem 1.3

Formulate the objective function. Make sure to include any needed
derivations or justifications for your equation(s).

The objective of this function is to minimize the total daily cost of the waste disposal system. The total daily cost would be composed of the fixed costs for operating the facilities in dollars per day, tipping costs in dollars per Mg, transportation costs in dollars per Mg-km, and recycling costs in dollars per Mg recycled.

$$ Total = Fixed + Tipping + Transportation + Recycling $$

The fixed cost F would be 2000 + 1500 + 2500 dollars per day. The tipping cost T would be the cost multiplied by the mass of waste entering each facility. The transportation cost Tr would be the distance from each city to facility multiplied by transportation cost multiplied by mass of waste being transported. The recycling cost Re would be the recycling processing cost multiplied by mass recycled R from each city. 

$$ F = 2000 + 1500 + 2500 $$
$$ T = \sum_{i,f}{t_f \times x_{i,f}} $$
$$ Tr = \sum_{i,f}{d_{i,f} \times 1.5 x_{i,f}}$$
$$ Re = \sum_i{R_i \times 40}$$
$$ Total = 6000 + \sum_{i,f}{t_f \times x_{i,f}} + \sum_{i,f}{d_{i,f} \times 1.5 x_{i,f}} + \sum_i{R_i \times 40}$$


#### Problem 1.4

Derive all relevant constraints. Make sure to include any needed
justifications or derivations.

The relevant constraints would be that disposal cannot exceed the maximum capacity of each facility due to system constraints. So LF cannot exceed 200 Mg, MRF cannot exceed 350 Mg, WTE cannot exceed 210 Mg. Additionally, the waste consumed and residuals produced from each city must match the amount disposed daily (e.g. 100, 90 and 120 for cities 1-3). Any of the relevant decision variables must be non-negative for conservation of mass. 

#### Problem 1.5

Find the optimal solution (using `JuMP` to solve the problem). Report
the optimal objective value.

We create the model with HiGHS Optimizer. Before creating the decision variables, data is formulated to set up the incoming supply from each facility, constant values for the rate of recycling, and recycling cost and transportation cost. An array is created to track the distance from each city to each facility. A separate dictionary is made to document the distances from facility to facility. The decision variables listed previously are made, followed by the design constraints. The objective function is also made based on the different cost components.  

In [3]:
# Build and solve the MSW allocation MILP
model = Model(HiGHS.Optimizer)

# Data
cities = [1, 2, 3]
supply = [100.0, 90.0, 120.0]  # Mg/day
facilities = ["LF", "MRF", "WTE"]
num_facilities = 3

cap = Dict("LF" => 200.0, "MRF" => 350.0, "WTE" => 210.0)
fixed = Dict("LF" => 2000.0, "MRF" => 1500.0, "WTE" => 2500.0)
tipping = Dict("LF" => 50.0, "MRF" => 7.0, "WTE" => 60.0)
recycle_rate = 0.40
recycle_proc_cost = 40.0
transport_cost_per_Mg_km = 1.5

# City to facility distances (km) - rows: cities 1,2,3; cols: LF, MRF, WTE
dist_cf = [5.0  30.0  15.0;
           15.0 25.0  10.0;
           13.0 45.0  20.0]

# Facility to facility distances (km) - for residual transport from MRF
dist_ff = Dict(("MRF", "LF") => 32.0, ("MRF", "WTE") => 15.0)

# Decision Variables
# x[i,j] = mass (Mg) from city i to facility j (1=LF, 2=MRF, 3=WTE)
@variable(model, x[i in cities, j in 1:num_facilities] >= 0)

# r[i,j] = residual mass from city i's MRF processing sent to facility j (1=LF, 3=WTE)
# r[i,1] = to LF, r[i,3] = to WTE
@variable(model, r[i in cities, j in [1, 3]] >= 0)

# y[j] = binary: 1 if facility j is open (1=LF, 2=MRF, 3=WTE)
@variable(model, y[j in 1:num_facilities], Bin)

# Constraints

# Supply balance: each city's waste must go somewhere (to LF, MRF, or WTE)
for i in cities
    @constraint(model, sum(x[i, j] for j in 1:num_facilities) == supply[i])
end

# MRF residual balance: non-recycled mass from MRF must go to LF or WTE
for i in cities
    @constraint(model, r[i, 1] + r[i, 3] == (1.0 - recycle_rate) * x[i, 2])
end

# Facility capacity constraints (with facility open indicator)
@constraint(model, sum(x[i, 1] for i in cities) + sum(r[i, 1] for i in cities) <= cap["LF"] * y[1]) # LF
@constraint(model, sum(x[i, 2] for i in cities) <= cap["MRF"] * y[2]) # MRF 
@constraint(model, sum(x[i, 3] for i in cities) + sum(r[i, 3] for i in cities) <= cap["WTE"] * y[3]) # WTE

# Objective: minimize total daily cost
# Fixed costs (incurred only if facility is open)
fixed_cost = 2000 * y[1] + 1500 * y[2] + 2500 * y[3]

# Tipping costs (applied to mass entering each facility)
tipping_cost = 50 * (sum(x[i, 1] for i in cities) + sum(r[i, 1] for i in cities)) +  # LF
               7 * sum(x[i, 2] for i in cities) +  # MRF
               60 * (sum(x[i, 3] for i in cities) + sum(r[i, 3] for i in cities))  # WTE

# Transportation costs: city to facility
transport_city_fac = sum(1.5 * dist_cf[i, j] * x[i, j] for i in cities, j in 1:num_facilities)

# Transportation costs: residual from MRF to LF or WTE
transport_residual = 1.5 * dist_ff[("MRF", "LF")] * sum(r[i, 1] for i in cities) +
                     1.5 * dist_ff[("MRF", "WTE")] * sum(r[i, 3] for i in cities)

# Recycling processing cost (applies to recycled mass from MRF)
recycle_cost = 40 * 0.40 * sum(x[i, 2] for i in cities)

@objective(model, Min, fixed_cost + tipping_cost + transport_city_fac + transport_residual + recycle_cost)

# Solve
optimize!(model)

Running HiGHS 1.12.0 (git hash: 755a8e027): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 9 rows; 18 cols; 36 nonzeros; 3 integer variables (3 binary)
Coefficient ranges:
  Matrix  [6e-01, 4e+02]
  Cost    [6e+01, 2e+03]
  Bound   [1e+00, 1e+00]
  RHS     [9e+01, 1e+02]
Presolving model
9 rows, 18 cols, 36 nonzeros  0s
6 rows, 15 cols, 33 nonzeros  0s
6 rows, 15 cols, 33 nonzeros  0s
Presolve reductions: rows 6(-3); columns 15(-3); nonzeros 33(-3) 

Solving MIP model with:
   6 rows
   15 cols (3 binary, 0 integer, 0 implied int., 12 continuous, 0 domain fixed)
   33 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B

We print the results of the model (created with assistance from GPT).

In [8]:
# Results
using Printf
obj_value = objective_value(model)
@printf("Optimal Total Daily Cost: %.2f USD\n\n", obj_value)

# Facility usage
@printf("LF open:  %d (capacity utilized: %.2f / 200.0 Mg)\n", Int(value(y[1])), 
        value(sum(x[i, 1] for i in cities) + sum(r[i, 1] for i in cities)))
@printf("MRF open: %d (capacity utilized: %.2f / 350.0 Mg)\n", Int(value(y[2])), 
        value(sum(x[i, 2] for i in cities)))
@printf("WTE open: %d (capacity utilized: %.2f / 210.0 Mg)\n\n", Int(value(y[3])), 
        value(sum(x[i, 3] for i in cities) + sum(r[i, 3] for i in cities)))

# Flows from cities
println("Detailed Flows (Mg/day):")
@printf("%-8s | %-10s | %-10s | %-10s\n", "City", "→ LF", "→ MRF", "→ WTE")
println(repeat("-", 45))
for i in cities
    @printf("City %d   | %10.2f | %10.2f | %10.2f\n", i, value(x[i, 1]), value(x[i, 2]), value(x[i, 3]))
end

println("\nMRF Residual Flows (Mg/day):")
@printf("%-8s | %-15s | %-15s\n", "City", "Residual → LF", "Residual → WTE")
println(repeat("-", 40))
for i in cities
    @printf("City %d   | %15.2f | %15.2f\n", i, value(r[i, 1]), value(r[i, 3]))
end

# Summary
println("\nTotal Facility Flows (Mg/day):")
total_lf = value(sum(x[i, 1] for i in cities) + sum(r[i, 1] for i in cities))
total_mrf = value(sum(x[i, 2] for i in cities))
total_wte = value(sum(x[i, 3] for i in cities) + sum(r[i, 3] for i in cities))
@printf("LF:  %.2f\n", total_lf)
@printf("MRF: %.2f\n", total_mrf)
@printf("WTE: %.2f\n", total_wte)

# Cost breakdown
println("\nCost Breakdown (USD):")
fixed_cost_val = value(fixed_cost);
tipping_cost_val = value(tipping_cost);
transport_city_val = value(transport_city_fac);
transport_resid_val = value(transport_residual);
recycle_cost_val = value(recycle_cost);

@printf("Fixed costs:              %.2f\n", fixed_cost_val)
@printf("Tipping costs:            %.2f\n", tipping_cost_val)
@printf("City→Facility transport:  %.2f\n", transport_city_val)
@printf("Residual transport:       %.2f\n", transport_resid_val)
@printf("Recycling processing:     %.2f\n", recycle_cost_val)

Optimal Total Daily Cost: 25750.00 USD

LF open:  1 (capacity utilized: 200.00 / 200.0 Mg)
MRF open: 0 (capacity utilized: 0.00 / 350.0 Mg)
WTE open: 1 (capacity utilized: 110.00 / 210.0 Mg)

Detailed Flows (Mg/day):
City     | → LF       | → MRF      | → WTE     
---------------------------------------------
City 1   |     100.00 |       0.00 |       0.00
City 2   |       0.00 |       0.00 |      90.00
City 3   |     100.00 |      -0.00 |      20.00

MRF Residual Flows (Mg/day):
City     | Residual → LF   | Residual → WTE 
----------------------------------------
City 1   |            0.00 |            0.00
City 2   |            0.00 |            0.00
City 3   |            0.00 |            0.00

Total Facility Flows (Mg/day):
LF:  200.00
MRF: 0.00
WTE: 110.00

Cost Breakdown (USD):
Fixed costs:              4500.00
Tipping costs:            16600.00
City→Facility transport:  4650.00
Residual transport:       0.00
Recycling processing:     0.00
LF open:  1 (capacity utilized: 200.00 /

#### Problem 1.6

Draw a diagram showing the flows of waste between the cities and the
facilities. Which facilities (if any) will not be used? Does this
solution make sense?

<img src="flow.png">

From the waste flow diagram, only the MRF facility is not used. Although this initially appears strange, this is because the MRF facility only recycles a fraction of the incoming waste. Non recyclable waste would still have to go through the LF or WTE facilities, adding extra transportation costs than just sending waste to those facilities in the first place. Sending waste to the MRF facility would only be prioritized if the recycling rate was higher or there are lower transportation costs to justify the recycling process.

### Problem 2 (6 points)

Consider a two-period economic dispatch problem, based on the
multi-period example from Lecture 14 (on 10/29). The generator data,
including ramping constraints for each generator, is provided in
\`data/generators.csv.’ In period 1, the demand is
$d_1 = 1100 \text{MW}$. In period 2, the demand is projected to be
$d_2 = 1200 \text{MW}$, but there is a 25% probability that it is \$1500
. In the first period, the solar capacity factor is $0.9$ and the wind
capacity factor is $0.45$, but in the second period, there is some
uncertainty: the forecasted solar and wind capacity factors are $0.95$
and $0.4$, respectively, but there is a 30% probability that they are
$0.75$ and $0.5$. Your goal is to identify how to dispatch your
generators to minimize the cost of meeting demand.

#### Problem 2.1

Draw a scenario tree for this problem.

For Period 1, demand is at 1100 MW with solar capacity factor of 0.9 and wind capacity factor of 0.45. These are guaranteed to happen.

For Period 2, there is a 75% probability that demand is at 1200 MW, and a 25% probability that demand is 1500 MW. Additionally, there is a 70% probability that solar and wind capacity factors are 0.95 and 0.4, and a 30% probability that those factors are 0.75 and 0.5.

<img src="scenario.png">

#### Problem 2.2

Formulate a stochastic linear program for this problem based on your
scenario tree from Problem 2.1 and the data in `data/generators.csv`.

Known:
- D1 MW (Period 1)
- D2A = 1200 MW (Period 2 scenario A)
- D2B = 1500 MW (Period 2 scenario B)

- Scenario A: $c_s^A = 0.95$ (solar), $c_w^A = 0.4$ (wind)
- Scenario B: $c_s^B = 0.75$ (solar), $c_w^B = 0.5$ (wind)
- Period 1 (deterministic): $c_s^1 = 0.9$, $c_w^1 = 0.45$

Generator data (from `generators.csv`):
- $P^{\min}_g$ = minimum power output (MW)
- $P^{\max}_g$ = maximum power capacity (MW)
- $c_g$ = variable cost (\$/MWh)
- $R_g$ = ramp rate (MW/hour, assuming 1-hour periods)

The decision variables would be the installed capacity of each generator type in MW, the energy produced from each generator in MWh, the amount of capacity installed from each generator in MWh similar to a single dispatch problem. It may be better to split the linear program into two based on the two periods. 

The objective function would be minimizing the total operational costs while ensuring that capacity is met. The total operational costs would be the operational costs from Period 1 + the expected average operational costs in Period 2 when considering all the possible scenarios. The operational costs would include the fixed and variable costs from each generator. 

The constraints consistent through both periods would be ensuring that the capacity limits for each generator are not exceeded in MW, there cannot be any negative values for capacity, and the demand balance must match the system requirements based on the period and scenario (1100 MW for Period 1, and either 1200MW or 1500MW for Period 2). Additionally, the ramping rates must be no greater than their maximum for each generator. The solar and wind ramping limits must be adjusted based on the specific capacity factors in each period/scenario. 

## References

Lecture Slides and ChatGPT.